# Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment) with Hyperparameter Tuning
This standalone notebook compares a Single-Layer Perceptron (PLA) implemented from scratch against a Tuned Multilayer Perceptron (MLP) on the 62-class English Handwritten Characters benchmark. All figures strictly adhere to Sharruk's ML Lab Guidelines (Times New Roman, 15 pt bold labels, 600 DPI vector EPS).

In [1]:
import os
import time
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, confusion_matrix
)

plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 15,
    'axes.labelsize': 15,
    'axes.labelweight': 'bold',
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 15,
    'figure.titlesize': 16,
    'figure.titleweight': 'bold'
})
warnings.filterwarnings('ignore')

def resolve_path(rel_path):
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    if os.path.basename(os.getcwd()) == 'Ex9':
        if rel_path.startswith('Ex9/'):
            return rel_path[len('Ex9/'):]
    return rel_path

os.makedirs(resolve_out('Ex9'), exist_ok=True)

In [2]:
class SingleLayerPLA:
    def __init__(self, n_classes, lr=0.01, max_epochs=30):
        self.n_classes = n_classes
        self.lr = lr
        self.max_epochs = max_epochs
        self.weights = None
        self.biases = None
        self.error_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros((self.n_classes, n_features))
        self.biases = np.zeros(self.n_classes)
        y_bin = label_binarize(y, classes=range(self.n_classes))
        for epoch in range(self.max_epochs):
            epoch_errors = 0
            for i in range(n_samples):
                xi = X[i]
                for c in range(self.n_classes):
                    pred = 1 if (np.dot(self.weights[c], xi) + self.biases[c]) >= 0 else 0
                    err = y_bin[i, c] - pred
                    if err != 0:
                        self.weights[c] += self.lr * err * xi
                        self.biases[c] += self.lr * err
                        epoch_errors += 1
            self.error_history.append((epoch_errors / (n_samples * self.n_classes)) * 100.0)
        return self

    def predict(self, X):
        return np.argmax(np.dot(X, self.weights.T) + self.biases, axis=1)

In [3]:
def run_experiment_9(data_dir='Datasets/English_Characters'):
    print('='*60)
    print('=== LAUNCHING EXPERIMENT 9: PLA vs MLP A/B PIPELINE ===')
    print('='*60)
    dir_path = Path(resolve_path(data_dir))
    df = pd.read_csv(dir_path / 'english.csv')
    images, labels = [], []
    for _, r in df.iterrows():
        fp = dir_path / r['image']
        if fp.exists():
            with Image.open(fp) as img:
                arr = np.array(img.convert('L').resize((28, 28)), dtype=np.float32) / 255.0
                images.append(arr.flatten())
                labels.append(str(r['label']))

    X = np.array(images)
    le = LabelEncoder()
    y = le.fit_transform(labels)
    n_classes = len(le.classes_)

    # 1. Generate 12-Panel EDA Plot (Sharruk's Rule 2)
    fig, axes = plt.subplots(3, 4, figsize=(20, 14))
    unique_cls, counts = np.unique(labels, return_counts=True)
    axes[0, 0].bar(range(15), counts[:15], color='#2b5c8f')
    axes[0, 0].set_title('Class Distribution (First 15)', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Class Index', fontsize=13, fontweight='bold')
    axes[0, 0].set_ylabel('Sample Count', fontsize=13, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].hist(X.ravel(), bins=30, color='#7570b3', edgecolor='black', alpha=0.7)
    axes[0, 1].set_title('Pixel Intensity Distribution', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Normalized Intensity', fontsize=13, fontweight='bold')
    axes[0, 1].set_ylabel('Pixel Frequency', fontsize=13, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)

    mean_2d = X.mean(axis=0).reshape(28, 28)
    axes[0, 2].plot(range(28), mean_2d.mean(axis=1), marker='o', color='#d95f02', linewidth=2)
    axes[0, 2].set_title('Mean Horizontal Profile', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('Row Index', fontsize=13, fontweight='bold')
    axes[0, 2].set_ylabel('Mean Intensity', fontsize=13, fontweight='bold')
    axes[0, 2].grid(True, alpha=0.3)

    axes[0, 3].plot(range(28), mean_2d.mean(axis=0), marker='s', color='#1b9e77', linewidth=2)
    axes[0, 3].set_title('Mean Vertical Profile', fontsize=14, fontweight='bold')
    axes[0, 3].set_xlabel('Col Index', fontsize=13, fontweight='bold')
    axes[0, 3].set_ylabel('Mean Intensity', fontsize=13, fontweight='bold')
    axes[0, 3].grid(True, alpha=0.3)

    pca = PCA(n_components=15, random_state=42)
    pca.fit(X)
    axes[1, 0].plot(range(1, 16), pca.explained_variance_ratio_ * 100, marker='D', color='#e7298a', linewidth=2)
    axes[1, 0].set_title('PCA Scree (Variance Ratio)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Principal Component', fontsize=13, fontweight='bold')
    axes[1, 0].set_ylabel('Variance (%)', fontsize=13, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(range(1, 16), np.cumsum(pca.explained_variance_ratio_) * 100, marker='^', color='#66a61e', linewidth=2)
    axes[1, 1].set_title('Cumulative PCA Variance', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Principal Component', fontsize=13, fontweight='bold')
    axes[1, 1].set_ylabel('Cumulative (%)', fontsize=13, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

    digit_mask = np.isin(labels, [str(i) for i in range(10)])
    upper_mask = np.isin(labels, [chr(c) for c in range(ord('A'), ord('Z')+1)])
    lower_mask = np.isin(labels, [chr(c) for c in range(ord('a'), ord('z')+1)])
    cats = ['Digits (0-9)', 'Upper (A-Z)', 'Lower (a-z)']
    cat_means = [X[digit_mask].mean(), X[upper_mask].mean(), X[lower_mask].mean()]
    axes[1, 2].bar(cats, cat_means, color=['#1f78b4', '#33a02c', '#fb9a99'])
    axes[1, 2].set_title('Mean Brightness by Category', fontsize=14, fontweight='bold')
    axes[1, 2].set_ylabel('Mean Pixel Value', fontsize=13, fontweight='bold')
    axes[1, 2].grid(axis='y', alpha=0.3)

    axes[1, 3].plot(range(28), mean_2d.var(axis=1), marker='v', color='#e6ab02', linewidth=2)
    axes[1, 3].set_title('Row-wise Pixel Variance', fontsize=14, fontweight='bold')
    axes[1, 3].set_xlabel('Row Index', fontsize=13, fontweight='bold')
    axes[1, 3].set_ylabel('Variance', fontsize=13, fontweight='bold')
    axes[1, 3].grid(True, alpha=0.3)

    digit_densities = [1.0 - X[labels == str(d)].mean() for d in range(10)]
    axes[2, 0].bar(range(10), digit_densities, color='#a6761d')
    axes[2, 0].set_title('Ink Density: Digits 0-9', fontsize=14, fontweight='bold')
    axes[2, 0].set_xlabel('Digit', fontsize=13, fontweight='bold')
    axes[2, 0].set_ylabel('Ink Fraction', fontsize=13, fontweight='bold')
    axes[2, 0].set_xticks(range(10))
    axes[2, 0].grid(axis='y', alpha=0.3)

    pixel_vars = X.var(axis=0)
    axes[2, 1].hist(pixel_vars, bins=25, color='#467821', edgecolor='black', alpha=0.7)
    axes[2, 1].set_title('Pixel Variance Distribution', fontsize=14, fontweight='bold')
    axes[2, 1].set_xlabel('Pixel Variance', fontsize=13, fontweight='bold')
    axes[2, 1].set_ylabel('Count (of 784)', fontsize=13, fontweight='bold')
    axes[2, 1].grid(True, alpha=0.3)

    top10_idx = np.argsort(pixel_vars)[-10:]
    axes[2, 2].bar(range(1, 11), pixel_vars[top10_idx], color='#8c564b')
    axes[2, 2].set_title('Top-10 Variable Pixels', fontsize=14, fontweight='bold')
    axes[2, 2].set_xlabel('Rank', fontsize=13, fontweight='bold')
    axes[2, 2].set_ylabel('Variance', fontsize=13, fontweight='bold')
    axes[2, 2].grid(True, alpha=0.3)

    std_2d = X.std(axis=0).reshape(28, 28)
    axes[2, 3].plot(range(28), std_2d.mean(axis=1), marker='P', color='#17becf', linewidth=2)
    axes[2, 3].set_title('Row-wise Std Deviation', fontsize=14, fontweight='bold')
    axes[2, 3].set_xlabel('Row Index', fontsize=13, fontweight='bold')
    axes[2, 3].set_ylabel('Mean Std Dev', fontsize=13, fontweight='bold')
    axes[2, 3].grid(True, alpha=0.3)

    plt.suptitle('Consolidated 12-Panel Exploratory Data Analysis: English Handwritten Characters', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(resolve_out('Ex9/EDA_Glyphs.eps'), format='eps', dpi=600)
    plt.close()

    # 2. Train Models & Loss Convergence
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    t0 = time.time()
    pla = SingleLayerPLA(n_classes=n_classes, lr=0.01, max_epochs=40).fit(X_tr, y_tr)
    pla_time = time.time() - t0
    pla_preds = pla.predict(X_te)

    t0 = time.time()
    mlp = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
                        learning_rate_init=0.001, max_iter=40, random_state=42).fit(X_tr, y_tr)
    mlp_time = time.time() - t0
    mlp_preds = mlp.predict(X_te)

    # Save Loss Convergence (600 DPI EPS)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    ax1.plot(range(1, len(pla.error_history) + 1), pla.error_history, marker='o', color='#d95f02', linewidth=2.5)
    ax1.set_title('Single-Layer PLA Error Rate vs. Epochs', fontsize=15, fontweight='bold')
    ax1.set_xlabel('Epoch', fontsize=15, fontweight='bold')
    ax1.set_ylabel('Step Error Rate (%)', fontsize=15, fontweight='bold')
    ax1.grid(True, alpha=0.3)

    ax2.plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_, marker='s', color='#2b5c8f', linewidth=2.5)
    ax2.set_title('MLP Cross-Entropy Loss vs. Epochs', fontsize=15, fontweight='bold')
    ax2.set_xlabel('Epoch', fontsize=15, fontweight='bold')
    ax2.set_ylabel('Cross-Entropy Loss', fontsize=15, fontweight='bold')
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Convergence Dynamics: PLA Linear Step vs. MLP Backpropagation', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(resolve_out('Ex9/Loss_Convergence.eps'), format='eps', dpi=600)
    plt.close()

    # Save ROC Curves (600 DPI EPS)
    y_te_bin = label_binarize(y_te, classes=range(n_classes))
    mlp_probs = mlp.predict_proba(X_te)
    plt.figure(figsize=(9, 7))
    for i in range(5):
        fpr, tpr, _ = roc_curve(y_te_bin[:, i], mlp_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, linewidth=2, label=f"Class '{le.classes_[i]}' (AUC = {roc_auc:.2f})")
    fpr_macro, tpr_macro, _ = roc_curve(y_te_bin.ravel(), mlp_probs.ravel())
    macro_auc = auc(fpr_macro, tpr_macro)
    plt.plot(fpr_macro, tpr_macro, 'k--', linewidth=2.5, label=f'Micro-Average (AUC = {macro_auc:.2f})')
    plt.plot([0, 1], [0, 1], ':', color='gray', linewidth=1.5)
    plt.title('Multilayer Perceptron ROC Curves (Representative Classes)', fontsize=16, fontweight='bold')
    plt.xlabel('False Positive Rate', fontsize=15, fontweight='bold')
    plt.ylabel('True Positive Rate', fontsize=15, fontweight='bold')
    plt.legend(loc='lower right', fontsize=13)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(resolve_out('Ex9/ROC_Curves.eps'), format='eps', dpi=600)
    plt.close()

    # Save Confusion Matrix (600 DPI EPS)
    subset_k = 15
    cm = confusion_matrix(y_te, mlp_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm[:subset_k, :subset_k], annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=le.classes_[:subset_k], yticklabels=le.classes_[:subset_k],
                annot_kws={'fontsize': 11})
    plt.title('MLP Confusion Matrix (First 15 Classes)', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Class', fontsize=15, fontweight='bold')
    plt.ylabel('True Class', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(resolve_out('Ex9/Confusion_Matrix.eps'), format='eps', dpi=600)
    plt.close()

    df_ab = pd.DataFrame([
        {
            'Model': 'Single-Layer Perceptron (PLA)',
            'Accuracy (%)': round(accuracy_score(y_te, pla_preds) * 100, 2),
            'Precision (%)': round(precision_score(y_te, pla_preds, average='weighted', zero_division=0) * 100, 2),
            'Recall (%)': round(recall_score(y_te, pla_preds, average='weighted', zero_division=0) * 100, 2),
            'F1-Score (%)': round(f1_score(y_te, pla_preds, average='weighted', zero_division=0) * 100, 2),
            'Time (s)': round(pla_time, 2)
        },
        {
            'Model': 'Tuned Multilayer Perceptron (MLP)',
            'Accuracy (%)': round(accuracy_score(y_te, mlp_preds) * 100, 2),
            'Precision (%)': round(precision_score(y_te, mlp_preds, average='weighted', zero_division=0) * 100, 2),
            'Recall (%)': round(recall_score(y_te, mlp_preds, average='weighted', zero_division=0) * 100, 2),
            'F1-Score (%)': round(f1_score(y_te, mlp_preds, average='weighted', zero_division=0) * 100, 2),
            'Time (s)': round(mlp_time, 2)
        }
    ])
    print('Saved all Ex9 figures at 600 DPI vector EPS.')
    print(df_ab.to_string(index=False))
    return df_ab

In [4]:
# Master Execution Cell
ex9_results = run_experiment_9()
print('=== EXPERIMENT 9 EXECUTION COMPLETE ===')

=== LAUNCHING EXPERIMENT 9: PLA vs MLP A/B PIPELINE ===


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved all Ex9 figures at 600 DPI vector EPS.
                            Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  Time (s)
    Single-Layer Perceptron (PLA)         14.96          26.76       14.96         13.47     20.44
Tuned Multilayer Perceptron (MLP)         24.49          24.04       24.49         22.16      6.83
=== EXPERIMENT 9 EXECUTION COMPLETE ===
